# Modelo Base de Recomendación y Ruta Greedy

Este notebook desarrolla una primera aproximación al problema de recomendación de POIs en Barcelona.

La finalidad de esta fase no es construir todavía un sistema híbrido completo, sino establecer una baseline sencilla, explicable y útil como punto de partida para el resto del TFM.

En concreto, aquí se trabajan dos ideas básicas:

- ranking simple de POIs a partir de variables ya disponibles en el dataset
- construcción de una ruta greedy basada en cercanía geográfica

Esta baseline servirá después para comparar y justificar por qué hacen falta módulos más avanzados como TF-IDF, clustering geográfico o la integración híbrida final.

## Objetivo del notebook

El flujo de este notebook sigue una lógica incremental y muy interpretable:

1. Cargar el dataset procesado
2. Revisar columnas y tipos relevantes
3. Construir una señal sencilla de ranking (`score_final`)
4. Probar una función básica de recomendación
5. Implementar una primera ruta greedy por proximidad
6. Obtener una referencia inicial para fases posteriores

La idea clave aquí es demostrar que ya se puede generar una recomendación razonable sin modelos complejos, aunque todavía con limitaciones claras.

## 1. Carga de librerías y datos

En esta primera sección se cargan las librerías necesarias y el dataset procesado del proyecto. También se muestra una vista inicial de los datos para comprobar que la carga se ha realizado correctamente.

In [ ]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic     # Importamos la función geodesic para calcular distancias entre coordenadas geográficas

# Cargamos el dataset en formato parquet
df = pd.read_parquet("../data/pois_barcelona_procesados.parquet")

# Mostramos las primeras filas del dataset para comprobar que los datos se han cargado correctamente
df.head()

## 2. Revisión estructural del dataset

Antes de definir ninguna lógica de recomendación, conviene comprobar que el dataset contiene las columnas necesarias y que los tipos de datos son coherentes con el uso posterior.

Esto es importante porque la baseline va a depender directamente de variables como:

- `category`
- `rating`
- `score`
- `latitude`
- `longitude`
- `visit_duration`

In [ ]:
# Verificamos que estám todas las columnas necesarias para realizar el modelado
df.columns

In [ ]:
# Comprobamos los tipos de las columnas para verificar que son los correctos
df.dtypes

## 3. Preparación de un subconjunto para la baseline

En esta fase se construye un DataFrame más manejable con solo las variables útiles para el modelo base. Además, se define una señal de ranking simple llamada `score_final`.

La lógica elegida es deliberadamente sencilla:

- 60% peso para `rating`
- 40% peso para `score`

No se pretende que esta combinación sea la definitiva, sino ofrecer una métrica inicial razonable para ordenar POIs antes de trabajar con modelos más sofisticados.

In [ ]:
# Creamos un nuevo DataFrame llamado df_base seleccionando solo las columnas relevantes  para simplificar el dataset y trabajar solo con lo necesario
df_base = df[[
    'id',
    'name',
    'category',
    'subcategory',
    'latitude',
    'longitude',
    'rating',
    'score',
    'visit_duration'
]].copy()   # hacemos una copia para no modificar el dataset original

# Eliminamos filas donde falten coordenadas (latitud o longitud) o porque luego vamos a calcular distancias geográficas
# Si hay valores NaN, la función geodesic falla
df_base = df_base.dropna(subset=['latitude', 'longitude'])

# Creamos una nueva columna llamada 'score_final' la cual será la métrica que usaremos para rankear los lugares
df_base['score_final'] = (
    df_base['rating'] * 0.6 +   # damos más peso al rating (60%)
    df_base['score'] * 0.4      # combinamos con el score interno (40%)
)

In [ ]:
# Comprobamos los valores de la columna de categoria para hacer pruebas posteriores
df["category"].value_counts()

## 4. Función básica de recomendación

La siguiente función representa la parte más simple del recomendador: filtra por categoría y rating mínimo, y después ordena los POIs según `score_final`.

Esto permite responder una pregunta muy básica pero útil:

> dados unos filtros sencillos, ¿cuáles son los POIs más prometedores según una señal de ranking inicial?

En esta etapa todavía no se tiene en cuenta la similitud semántica ni la coherencia global de una ruta.

In [ ]:
# Esta función sirve para obtener los mejores POIs según ciertos filtros
def recomendar(df, categoria=None, min_rating=0, top_n=10):
    
    # Hacemos una copia del DataFrame para no modificar el original
    data = df.copy()
    
    # Si el usuario especifica una categoría (por ejemplo "cultural") filtramos solo los POIs de esa categoría
    if categoria:
        data = data[data['category'] == categoria]
    
    # Filtramos los POIs que tengan un rating mínimo para evitar lugares con baja calidad
    data = data[data['rating'] >= min_rating]
    
    # Ordenamos los resultados por la columna 'score_final' de mayor a menor
    # Luego nos quedamos con los primeros N resultados (top_n)
    return data.sort_values(by='score_final', ascending=False).head(top_n)

# Ejecutamos la función (PRIMERA PRUEBA):
# - Solo POIs de categoría "cultural"
# - Con rating mínimo de 4
# - Mostramos los 5 mejores
recomendar(df_base, categoria='cultural', min_rating=4, top_n=5)

## 5. Primera ruta greedy por cercanía

Una vez resuelta la parte de ranking básico, se introduce una lógica secuencial para ordenar POIs geográficamente.

La estrategia es greedy:

- se parte de un punto inicial
- en cada paso se elige el POI más cercano al punto actual
- se repite el proceso hasta alcanzar el número deseado de POIs

Esta aproximación es útil como baseline porque es fácil de explicar, rápida de ejecutar y permite observar de forma clara las limitaciones que luego resolverán los módulos más avanzados.

In [ ]:
# Función para generar una ruta greedy
# El algoritmo parte de un punto inicial y en cada paso elige el POI más cercano
def generar_ruta(df, start_point, n=5):
    
    # Hacemos una copia del DataFrame para no modificar el original
    puntos = df.copy()
    
    # Eliminamos filas sin coordenadas válidas
    puntos = puntos.dropna(subset=['latitude', 'longitude'])
    
    # Lista donde guardaremos los POIs seleccionados
    ruta = []
    
    # Punto actual de partida
    actual = start_point
    
    # Repetimos hasta seleccionar n puntos o hasta quedarnos sin POIs
    for _ in range(min(n, len(puntos))):
        
        # Calculamos la distancia desde el punto actual hasta cada POI
        # Esta distancia es la que usa el algoritmo greedy para decidir el siguiente punto
        puntos['dist_actual_km'] = puntos.apply(
            lambda x: geodesic(actual, (x['latitude'], x['longitude'])).km,
            axis=1
        )
        
        # Calculamos la distancia desde el punto inicial del usuario hasta cada POI
        puntos['dist_inicio_km'] = puntos.apply(
            lambda x: geodesic(start_point, (x['latitude'], x['longitude'])).km,
            axis=1
        )
        
        # Seleccionamos el POI más cercano al punto actual
        siguiente = puntos.sort_values('dist_actual_km').iloc[0].copy()
        
        # Guardamos la distancia respecto al punto anterior de la ruta
        siguiente['dist_anterior_km'] = siguiente['dist_actual_km']
        
        # Añadimos el punto seleccionado a la ruta
        ruta.append(siguiente)
        
        # Actualizamos el punto actual con las coordenadas del POI elegido
        actual = (siguiente['latitude'], siguiente['longitude'])
        
        # Eliminamos el POI ya seleccionado para no repetirlo
        puntos = puntos.drop(siguiente.name)
    
    # Convertimos la lista de resultados en un DataFrame
    ruta_df = pd.DataFrame(ruta)
    
    # Si no hay resultados, devolvemos el DataFrame vacío
    if ruta_df.empty:
        return ruta_df
    
    # Calculamos distancia acumulada en km
    ruta_df['dist_acumulada_km'] = ruta_df['dist_anterior_km'].cumsum()
    
    # Convertimos todo a metros (numérico, mejor para ML)
    ruta_df['dist_desde_inicio_m'] = ruta_df['dist_inicio_km'] * 1000
    ruta_df['dist_desde_anterior_m'] = ruta_df['dist_anterior_km'] * 1000
    ruta_df['dist_acumulada_m'] = ruta_df['dist_acumulada_km'] * 1000
    
    # En la primera fila no hay punto anterior real
    ruta_df.loc[ruta_df.index[0], 'dist_desde_anterior_m'] = 0
    
    # Reordenamos columnas para que el resultado sea más claro
    columnas_finales = [
        'id',
        'name',
        'category',
        'subcategory',
        'rating',
        'score_final',
        'visit_duration',
        'latitude',
        'longitude',
        'dist_desde_anterior_m',
        'dist_desde_inicio_m',
        'dist_acumulada_m'
    ]
    
    # Nos quedamos solo con las columnas que existan en el DataFrame
    columnas_finales = [col for col in columnas_finales if col in ruta_df.columns]
    
    return ruta_df[columnas_finales]


# Punto inicial (ejemplo: centro de Barcelona)
start = (41.3851, 2.1734)

# Filtramos solo los POIs culturales con rating mínimo de 4
df_cultural = df_base[
    (df_base['category'] == 'cultural') &
    (df_base['rating'] >= 4.0)
].copy()

# Generamos una ruta de 5 POIs
ruta_cultural = generar_ruta(df_cultural, start, n=5)

# Reseteamos el índice para que empiece limpio
ruta_cultural = ruta_cultural.reset_index(drop=True)
ruta_cultural.index = ruta_cultural.index + 1

# Mostramos el resultado final
ruta_cultural

## 6. Comprobación manual de un POI concreto

La última consulta del notebook sirve como verificación puntual para inspeccionar un ejemplo concreto del dataset. Este tipo de comprobación es útil para validar manualmente nombres, categorías o registros de interés durante la fase exploratoria del modelo base.

In [ ]:
df[df['name'].str.contains("Sagrada", case=False, na=False)]

## Interpretación de esta fase

Este modelo base permite extraer varias ideas importantes:

- sí es posible generar recomendaciones iniciales con reglas simples
- sí es posible construir una ruta básica a partir de cercanía geográfica
- pero la calidad global de la ruta todavía depende demasiado de criterios locales y simples

En particular, esta baseline no resuelve todavía aspectos como:

- similitud temática entre POIs
- coherencia semántica de la ruta
- equilibrio entre relevancia y proximidad
- restricciones más realistas de distancia o tiempo
- continuidad espacial más allá del vecino más cercano

## Limitaciones del notebook

Aunque esta baseline es útil como punto de partida, conviene dejar claras sus limitaciones:

- `score_final` es una combinación heurística simple
- la ruta greedy solo mira cercanía local y no optimiza la secuencia completa
- no se usa todavía información textual para medir similitud
- no se emplea todavía una señal espacial agregada como `cluster_geo`
- no existe todavía una capa híbrida que combine varias señales a la vez

Estas limitaciones justifican el desarrollo de los notebooks posteriores.

## Conclusión del notebook

La decisión metodológica de esta fase es mantener este notebook como baseline del proyecto.

Su valor no está en ser el modelo final, sino en ofrecer una referencia clara, explicable y comparativa para demostrar la evolución del sistema:

- desde reglas simples
- hacia recomendación basada en contenido
- clustering geográfico
- ranking consolidado
- y finalmente el sistema híbrido de recomendación y creación de rutas